In [1]:
from tkinter.constants import CENTER

import cv2
import numpy as np

In [10]:
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("웹캠을 열 수 없습니다.")
while True:
    (ret, frame) = cap.read()
    if not ret:
        print("프레임 오류")
        break
    flip_frame = cv2.flip(frame, 1)
    (height, width, _) = flip_frame.shape
    (center_x, center_y) = (width // 2, height // 2)
    roi = flip_frame[center_y - 150:center_y + 150, center_x - 150:center_x + 150]
    cv2.rectangle(flip_frame,(center_x - 150, center_y - 150), (center_x + 150, center_y + 150), (0, 0, 255), 2)
    cv2.imshow("Webcam", flip_frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord("c" or "C"):
        gray_image = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        gray_image = np.flip(gray_image, axis=1)
        cv2.imwrite('gray_image.png', gray_image)
        gaussian_blur = cv2.GaussianBlur(gray_image, (5, 5), 3)
        (_, otsu_thresh)= cv2.threshold(gaussian_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        cv2.imshow("otsu_thresh", otsu_thresh)

        kernel = np.ones((5,5), np.uint8)
        erosion = cv2.erode(otsu_thresh, kernel, iterations=3)
        cv2.imshow("erosion", erosion)
        cv2.imwrite("digit_binary_image.png", erosion)
        img = cv2.imread("digit_binary_image.png", cv2.IMREAD_UNCHANGED)

        (h, w ) = img.shape[:2]
        crop_size = 300
        (cx, cy) = w // 2, h // 2
        half = crop_size // 2
        (x1, x2) = (cx - half, cx + half)
        (y1, y2) = (cy - half, cy + half)
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(w, x2)
        y2 = min(h, y2)

        crop = img[y1:y2, x1:x2]
        cv2.imshow("crop", crop)
        reversed_image = cv2.bitwise_not(crop)
        cv2.imshow("reversed_image", reversed_image)
        cv2.imwrite("image_for_test.png", reversed_image)
    if cv2.waitKey(30) == 27:
        break
cap.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 

In [2]:
# ===== MNIST 손글씨 숫자 데이터로 학습 =====
import keras
import matplotlib.pyplot as plt

mnist = keras.datasets.mnist
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# 픽셀값을 0~1 범위로 정규화
X_train = X_train / 255.0
X_test = X_test / 255.0
print('학습:', X_train.shape, ' 테스트:', X_test.shape)

학습: (60000, 28, 28)  테스트: (10000, 28, 28)


In [3]:

model = keras.Sequential(name='digit_classifier')
model.add(keras.layers.Input(shape=(28, 28)))
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(512, activation='relu'))
model.add(keras.layers.Dropout(0.2))
model.add(keras.layers.Dense(10, activation='softmax'))
model.summary()

Model: "digit_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       401,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         5,130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 407,050 (1.55 MB)

 Trainable params: 407,050 (1.55 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# 컴파일 → 학습 → 평가 → 저장
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.fit(X_train, y_train, epochs=10)

test_loss, test_acc = model.evaluate(X_test, y_test)
print('테스트 정확도:', test_acc)

model.save('digit_model.keras')   # 저장 시 확장자 .keras 필수

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9350 - loss: 0.2203
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9703 - loss: 0.0967
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9785 - loss: 0.0694
Epoch 4/10
1283/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9828 - loss: 0.0524

In [ ]:
# ===== 웹캠으로 저장한 숫자 예측 =====
# image_for_test.png 는 '흰 글씨 / 검은 배경'(MNIST 형식)으로 저장됨 → 추가 반전 불필요
test_img = cv2.imread('image_for_test.png', cv2.IMREAD_GRAYSCALE)
test_img = cv2.resize(test_img, (28, 28))
test_img = test_img.astype('float32') / 255.0

plt.imshow(test_img, cmap='gray')
plt.title('input image')
plt.show()

pred = model.predict(test_img.reshape(1, 28, 28))
print('예측 숫자 =', pred.argmax())
print('확률 = %.3f' % pred.max())